# 16 — Evaluations (Vertex AI Gen AI Evaluation Service)

**AWS-side equivalent:** Bedrock Online Evaluation + custom evaluators
(`notebooks/aws/16-evaluations.ipynb`)
**GCP service:** Vertex AI Gen AI Evaluation Service —
`vertexai.evaluation.{EvalTask, PointwiseMetric, PairwiseMetric, MetricPromptTemplateExamples}`.

_As of 2026-05: the stable import path is `vertexai.evaluation` (the older
`vertexai.preview.evaluation` is still aliased but deprecated in 1.71+). Install with
`pip install google-cloud-aiplatform[evaluation]`. The judge model defaults to
`gemini-2.5-flash` — we keep that, with `gemini-2.5-pro` for reasoning under test._

Flow mirrors the AWS notebook:
1. Build a small EvalDataset (5 prompts + reference answers).
2. Pointwise eval on the candidate model (groundedness + helpfulness).
3. Pairwise SxS between two model versions.
4. Persist the run id into `config.json`.

## Step 1 — Load config and init Vertex AI

In [ ]:
import json
import time
from pathlib import Path

import vertexai

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
STAGING_BUCKET = gcp.get('staging_bucket')
PREFIX = gcp.get('prefix', 'orgiq')
REASONING_MODEL = gcp.get('reasoning_model', 'gemini-2.5-pro')
FAST_MODEL = gcp.get('fast_model', 'gemini-2.5-flash')
BASELINE_MODEL = gcp.get('baseline_model', 'gemini-2.5-flash')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
assert STAGING_BUCKET, 'gcp.staging_bucket required — eval writes artefacts to GCS.'

vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)
print(f'✓ Vertex AI initialised — project={PROJECT_ID} region={REGION}')
print(f'  candidate={REASONING_MODEL}  baseline={BASELINE_MODEL}  judge=gemini-2.5-flash')

## Step 2 — Build the EvalDataset

Five hand-crafted OrgIQ cases that mirror the AWS notebook's `harness_cases`. Each row has
`prompt`, `reference` (the gold answer), `context` (graph snippets that the candidate
should ground in), and a `human_score` placeholder. The Evaluation Service builds a
DataFrame from this dict.

Drop in `data/org/harness_cases.json` later if you want to swap from hardcoded to file-based.

In [ ]:
import pandas as pd

eval_cases = [
    {
        'prompt': 'Who is the top Kubernetes expert at the company?',
        'reference': 'Alice Chen is the top Kubernetes expert with composite score 0.74.',
        'context': 'Alice Chen (p_alice) composite_score=0.74, has_skill Kubernetes weight=0.92.',
        'human_score': 5,
    },
    {
        'prompt': 'Tell me about Eve Johnson\'s expertise.',
        'reference': 'Eve Johnson is a security engineer with Authentication (0.95) and Microservices (0.6).',
        'context': 'Eve Johnson (p_eve) title=Security Engineer; skills: Authentication 0.95, Microservices 0.6.',
        'human_score': 5,
    },
    {
        'prompt': 'Who can help with CI/CD?',
        'reference': 'Dave Park has strong CI/CD expertise (0.9). Alice Chen also knows CI/CD (0.8).',
        'context': 'Dave Park (p_dave) has_skill CI/CD weight=0.9; Alice Chen has_skill CI/CD weight=0.8.',
        'human_score': 4,
    },
    {
        'prompt': 'What documents discuss microservices architecture?',
        'reference': 'Service Mesh Patterns authored by Carol Diaz.',
        'context': 'Document "Service Mesh Patterns" authored by Carol Diaz (p_carol). Skill: Microservices 0.9.',
        'human_score': 4,
    },
    {
        'prompt': 'Who collaborates most with Alice Chen?',
        'reference': 'Bob Kumar — collaboration weight 0.93.',
        'context': 'collaborations(p_alice, p_bob, weight=0.93). No other strong edges.',
        'human_score': 5,
    },
]

eval_df = pd.DataFrame(eval_cases)
print(f'✓ eval dataset: {len(eval_df)} rows')
eval_df

## Step 3 — Generate candidate responses

EvalTask can either receive a pre-computed `response` column or take a `model` argument and
run inference itself. We do the former so we can stream the same generations into both the
pointwise and pairwise tasks below.

In [ ]:
from google import genai

# google-genai client; auto-discovers Vertex AI from env (VERTEXAI=true) or explicit args.
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)


def _generate(model_id: str, prompt: str, context: str) -> str:
    resp = genai_client.models.generate_content(
        model=model_id,
        contents=[
            f'Use only the context below to answer.\n\n'
            f'CONTEXT:\n{context}\n\n'
            f'QUESTION: {prompt}'
        ],
    )
    return resp.text or ''


eval_df['response'] = [
    _generate(REASONING_MODEL, row.prompt, row.context) for row in eval_df.itertuples()
]
eval_df['baseline_response'] = [
    _generate(BASELINE_MODEL, row.prompt, row.context) for row in eval_df.itertuples()
]

print(f'✓ generated {len(eval_df)} candidate + baseline responses')
for i, row in eval_df.iterrows():
    print(f'\n[{i}] {row.prompt[:50]}')
    print(f'   candidate: {row.response[:100]}')
    print(f'   baseline:  {row.baseline_response[:100]}')

## Step 4 — Pointwise eval (groundedness + helpfulness)

Both metrics are pre-built in `MetricPromptTemplateExamples.Pointwise`. They use a
`gemini-2.5-flash` judge under the hood; cost ≈ \$0.001 per case.

EvalTask reads four columns by default:

| Column | Meaning |
|---|---|
| `prompt` | user input |
| `response` | candidate answer (what we're grading) |
| `reference` | gold answer (for metrics that need it) |
| `context` | grounding snippets (for groundedness) |

In [ ]:
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

pointwise_metrics = [
    MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS,
    MetricPromptTemplateExamples.Pointwise.HELPFULNESS,
]

pointwise_task = EvalTask(
    dataset=eval_df,
    metrics=pointwise_metrics,
    experiment=f'{PREFIX}-pointwise',
)

pointwise_result = pointwise_task.evaluate(
    experiment_run_name=f'{PREFIX}-pointwise-{int(time.time())}'
)

print('✓ pointwise eval complete')
print('\n— summary metrics —')
print(json.dumps(pointwise_result.summary_metrics, indent=2, default=str))
print('\n— per-row scores —')
score_cols = [c for c in pointwise_result.metrics_table.columns if c.endswith('/score')]
display_cols = ['prompt'] + score_cols
pointwise_result.metrics_table[display_cols]

## Step 5 — Pairwise SxS (candidate vs baseline)

Pairwise metrics compare two responses side-by-side using a judge. We use the prebuilt
`Pairwise.QUESTION_ANSWERING_QUALITY` so the judge sees both `response` (candidate) and
`baseline_model_response` (baseline) columns.

EvalTask convention: pairwise metrics read the candidate from `response` and the baseline
from `baseline_model_response`. Rename the column we generated earlier.

In [ ]:
from vertexai.evaluation import PairwiseMetric

pairwise_df = eval_df.rename(columns={'baseline_response': 'baseline_model_response'})

pairwise_metric = PairwiseMetric(
    metric='pairwise_question_answering_quality',
    metric_prompt_template=MetricPromptTemplateExamples.Pairwise.QUESTION_ANSWERING_QUALITY.metric_prompt_template,
)

pairwise_task = EvalTask(
    dataset=pairwise_df,
    metrics=[pairwise_metric],
    experiment=f'{PREFIX}-pairwise',
)

pairwise_result = pairwise_task.evaluate(
    experiment_run_name=f'{PREFIX}-pairwise-{int(time.time())}'
)

print('✓ pairwise eval complete')
print('\n— summary metrics —')
print(json.dumps(pairwise_result.summary_metrics, indent=2, default=str))
print('\n— per-row preference —')
pref_cols = [c for c in pairwise_result.metrics_table.columns
             if 'pairwise_choice' in c or c.endswith('/explanation')]
pairwise_result.metrics_table[['prompt'] + pref_cols].head()

## Step 6 — Compare candidate vs human scores

If `human_score` is present we can sanity-check the auto-eval against the hand-graded
ground truth. This is what AWS-side Bedrock Online Evaluation does via its
`Correctness` evaluator.

In [ ]:
comparison = pointwise_result.metrics_table[
    ['prompt', 'groundedness/score', 'helpfulness/score']
].copy()
comparison['human_score'] = eval_df['human_score'].values
comparison['auto_avg'] = (
    comparison['groundedness/score'] + comparison['helpfulness/score']
) / 2.0
comparison['delta_vs_human'] = comparison['auto_avg'] - comparison['human_score']
comparison

## Step 7 — Persist experiment ids into config.json

In [ ]:
config = json.loads(Path('config.json').read_text())
config.setdefault('gcp', {}).setdefault('eval', {})
config['gcp']['eval']['pointwise_experiment'] = f'{PREFIX}-pointwise'
config['gcp']['eval']['pairwise_experiment'] = f'{PREFIX}-pairwise'
config['gcp']['eval']['last_run_ts'] = int(time.time())
config['gcp']['eval']['candidate_model'] = REASONING_MODEL
config['gcp']['eval']['baseline_model'] = BASELINE_MODEL
Path('config.json').write_text(json.dumps(config, indent=2))
print('✓ config.gcp.eval written')

## Step 8 — Custom LLM-as-judge with your own rubric

The built-in `MetricPromptTemplateExamples` cover broad axes (groundedness,
helpfulness, etc.) but for production runs you often want a domain-specific
judge. This mirrors the AWS prompt-canary judge (`infra/lib/prompt-canary-stack.ts`)
which scores responses on **helpfulness / factuality / tone / conciseness**.

Pattern: one `PointwiseMetric` per axis, each backed by a custom
`PointwiseMetricPromptTemplate`. Judge model is `gemini-2.5-flash` (cheap, fast)
by default — Vertex AI uses it for evaluator calls when you don't override.

Why one metric per axis (vs one judge that returns 4 scores in JSON):
Vertex AI's evaluation framework produces a separate score column per metric
(`custom_helpfulness/score`, `custom_factuality/score`, …) which is much easier
to aggregate, threshold, and chart per-axis. Same total judge cost.

In [ ]:
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

# --- rubric: four axes, each scored 1-5 ---
RUBRIC = {
    'helpfulness': {
        'criteria': 'Does the response answer the question directly and completely?',
        'rubric': {
            '1': 'Does not answer the question.',
            '2': 'Partial answer with significant gaps.',
            '3': 'Adequate answer; would satisfy a casual reader.',
            '4': 'Strong, complete answer.',
            '5': 'Comprehensive, expert-level answer with helpful context.',
        },
    },
    'factuality': {
        'criteria': 'Does the response stay grounded in the implicit context (no fabrication)?',
        'rubric': {
            '1': 'Significant fabrication or contradicts the context.',
            '2': 'Several questionable claims.',
            '3': 'Mostly grounded, minor speculation.',
            '4': 'Well-grounded with limited inference.',
            '5': 'Fully grounded in the provided context.',
        },
    },
    'tone': {
        'criteria': 'Is the response professional and clear?',
        'rubric': {
            '1': 'Confusing or unprofessional.',
            '2': 'Awkward phrasing in multiple places.',
            '3': 'Mostly clear with minor issues.',
            '4': 'Clear and professional.',
            '5': 'Polished, well-structured, easy to read.',
        },
    },
    'conciseness': {
        'criteria': 'Is the response an appropriate length without filler?',
        'rubric': {
            '1': 'Severely under- or over-length; major filler.',
            '2': 'Noticeable padding or omissions.',
            '3': 'Reasonable length with some inefficiency.',
            '4': 'Well-paced, minimal filler.',
            '5': 'Optimal length and information density.',
        },
    },
}

custom_metrics = []
for axis, spec in RUBRIC.items():
    template = PointwiseMetricPromptTemplate(
        criteria={axis.title(): spec['criteria']},
        rating_rubric=spec['rubric'],
        input_variables=['prompt'],  # the judge sees prompt + response (response is auto-injected)
    )
    custom_metrics.append(
        PointwiseMetric(metric=f'custom_{axis}', metric_prompt_template=template)
    )

print(f'✓ defined {len(custom_metrics)} custom metrics: ' + ', '.join(m.metric for m in custom_metrics))

## Step 9 — Run as a tracked job (Vertex AI Experiments)

Bedrock kicks off evaluation as a *Job* — async, persistent, retrievable by id.
Vertex AI's equivalent is `EvalTask(experiment=..., experiment_run_name=...)`. Each
`evaluate()` call writes its run into Vertex AI Experiments under your experiment
name; you can list runs, fetch one by id, and compare runs across time.

Two run knobs that matter for jobs:
- `experiment` — stable group name (e.g. `orgiq-llm-judge-prod`)
- `experiment_run_name` — per-run id (timestamp + git sha is a good convention)

Below: kick off a job-style run that emits all four custom scores, then show
how to re-fetch its results via the experiment id.

In [ ]:
import datetime
import os

EVAL_EXPERIMENT = config['gcp'].setdefault('eval_experiment', 'orgiq-llm-judge')
EVAL_RUN_NAME = f"run-{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dt%H%M%S')}"

# Re-use the candidate responses computed in Step 3.
job_task = EvalTask(
    dataset=eval_df,
    metrics=custom_metrics,
    experiment=EVAL_EXPERIMENT,
)

print(f'→ kicking off job experiment={EVAL_EXPERIMENT} run={EVAL_RUN_NAME}')
t0 = time.monotonic()
job_result = job_task.evaluate(
    experiment_run_name=EVAL_RUN_NAME,
)
dt = time.monotonic() - t0
print(f'✓ job complete in {dt:.1f}s — {len(job_result.metrics_table)} rows scored')

# Per-axis summary
score_cols = [c for c in job_result.metrics_table.columns if c.endswith('/score')]
summary = job_result.metrics_table[score_cols].mean().round(3).to_dict()
print('per-axis mean scores:')
for k, v in summary.items():
    print(f'  {k:35s} {v}')

config['gcp']['eval']['last_job_experiment'] = EVAL_EXPERIMENT
config['gcp']['eval']['last_job_run'] = EVAL_RUN_NAME
config['gcp']['eval']['last_job_summary'] = summary
Path('config.json').write_text(json.dumps(config, indent=2))

### Retrieving a past job

Once a job has been written under an experiment id, you can list/get it later
via `google.cloud.aiplatform.Experiment` — no need to keep the original
`EvalTask` object around.

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)
exp = aiplatform.Experiment(experiment_name=EVAL_EXPERIMENT)
runs = list(exp.list_runs())
print(f'experiment {EVAL_EXPERIMENT!r} has {len(runs)} run(s):')
for r in runs[-5:]:
    print(f'  {r.name:35s}  state={r.state}')

## Step 10 — Persist results to BigQuery for trend analysis

Vertex AI Experiments keeps run history but isn't great for SQL-style trend
queries ("show me last-30-day judge drift on factuality"). The canonical
pattern is to also write each row of `metrics_table` into a BigQuery table
keyed by (experiment, run, prompt_hash, ts). Then text-to-SQL (notebook 18b)
can answer trend questions directly.

Schema: one row per (run, evaluated prompt) with one column per axis score.

In [ ]:
from google.cloud import bigquery
import hashlib

BQ_DATASET = config['gcp'].get('bq_dataset', 'orgiq')
BQ_LOCATION = config['gcp'].get('bq_location', 'US')
EVAL_TABLE = 'eval_runs'

bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

# Idempotent table create.
schema = [
    bigquery.SchemaField('ts', 'TIMESTAMP', mode='REQUIRED'),
    bigquery.SchemaField('experiment', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('run_name', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('prompt_hash', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('prompt', 'STRING'),
    bigquery.SchemaField('response', 'STRING'),
    bigquery.SchemaField('helpfulness', 'FLOAT64'),
    bigquery.SchemaField('factuality', 'FLOAT64'),
    bigquery.SchemaField('tone', 'FLOAT64'),
    bigquery.SchemaField('conciseness', 'FLOAT64'),
    bigquery.SchemaField('model_id', 'STRING'),
    bigquery.SchemaField('judge_model_id', 'STRING'),
]
table_id = f'{PROJECT_ID}.{BQ_DATASET}.{EVAL_TABLE}'
table = bigquery.Table(table_id, schema=schema)
table.time_partitioning = bigquery.TimePartitioning(field='ts')
table.clustering_fields = ['experiment', 'run_name']
bq.create_table(table, exists_ok=True)
print(f'✓ table ready: {table_id}')

# Project the latest job's metrics_table into the table.
now_ts = datetime.datetime.now(datetime.timezone.utc)
rows = []
for _, r in job_result.metrics_table.iterrows():
    prompt_text = str(r.get('prompt', ''))
    rows.append({
        'ts': now_ts.isoformat(),
        'experiment': EVAL_EXPERIMENT,
        'run_name': EVAL_RUN_NAME,
        'prompt_hash': hashlib.sha256(prompt_text.encode()).hexdigest()[:12],
        'prompt': prompt_text,
        'response': str(r.get('response', '')),
        'helpfulness':  float(r.get('custom_helpfulness/score', float('nan'))) if r.get('custom_helpfulness/score') is not None else None,
        'factuality':   float(r.get('custom_factuality/score', float('nan'))) if r.get('custom_factuality/score') is not None else None,
        'tone':         float(r.get('custom_tone/score', float('nan'))) if r.get('custom_tone/score') is not None else None,
        'conciseness':  float(r.get('custom_conciseness/score', float('nan'))) if r.get('custom_conciseness/score') is not None else None,
        'model_id':       config['gcp'].get('reasoning_model', 'gemini-2.5-pro'),
        'judge_model_id': config['gcp'].get('fast_model', 'gemini-2.5-flash'),
    })

errors = bq.insert_rows_json(table_id, rows)
if errors:
    print(f'✗ insert errors: {errors}')
else:
    print(f'✓ inserted {len(rows)} rows into {table_id}')

## Step 11 — Trend query: judge drift on a single axis over time

Once multiple runs land in `eval_runs` you can chart drift per-axis with one SQL.
Here's the pattern; with only one run so far it'll just return today's snapshot.

In [ ]:
trend_sql = f'''
SELECT
  DATE(ts) AS run_date,
  AVG(helpfulness)  AS avg_helpfulness,
  AVG(factuality)   AS avg_factuality,
  AVG(tone)         AS avg_tone,
  AVG(conciseness)  AS avg_conciseness,
  COUNT(*) AS n
FROM `{PROJECT_ID}.{BQ_DATASET}.{EVAL_TABLE}`
WHERE experiment = @experiment
GROUP BY run_date
ORDER BY run_date DESC
LIMIT 30
'''

job = bq.query(
    trend_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter('experiment', 'STRING', EVAL_EXPERIMENT)],
    ),
)
trend_df = job.to_dataframe()
print(f'\nLast {len(trend_df)} day(s) of eval runs for {EVAL_EXPERIMENT!r}:')
print(trend_df.to_string(index=False))

## Summary

- 5-row pandas EvalDataset built with prompt / reference / context / human_score.
- Candidate (`gemini-2.5-pro`) and baseline (`gemini-2.5-flash`) responses generated via
  the unified `google-genai` SDK against Vertex AI.
- Pointwise eval ran `MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS` and
  `.HELPFULNESS` through `EvalTask`.
- Pairwise SxS ran `Pairwise.QUESTION_ANSWERING_QUALITY` between candidate and baseline.
- Auto-eval scores compared against the `human_score` ground truth.
- Experiment ids persisted to `config.gcp.eval` so 17-cloud-trace can join eval scores
  against trace spans.

**Next:** [17-cloud-trace.ipynb](17-cloud-trace.ipynb) — Cloud Trace + OpenInference span projection.
- Custom LLM-as-judge with a 4-axis rubric (helpfulness / factuality / tone / conciseness)
  mirroring the AWS prompt-canary judge.
- Job-style runs via `EvalTask(experiment=…, experiment_run_name=…)` — listable and
  retrievable later through `google.cloud.aiplatform.Experiment`.
- Each run also lands in BigQuery `eval_runs` (time-partitioned, clustered by
  experiment+run) for trend analysis via SQL — including text-to-SQL in 18b.
